# Solo Enterprise agentgateway across two clusters, end to end: ingress, cross-cluster failover, and MCP tool federation over HBONE

Two ambient kind clusters (`east-ag` + `west-ag`) are already peered over HBONE (the `agentgw-multi-cluster-kind` standup). This notebook drives the **exercises** on top of that platform: expose a service through the **agentgateway ingress**, fail a service over to the **other cluster** with no client change, then federate **MCP tool servers** from both clusters behind **one** agentgateway endpoint and govern them with a single policy.

> **Kernel:** pick **Bash** (top-right). The standup (`scripts/quick.sh`) must already be green — this notebook only runs the exercises, not the setup.

## What you're running

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 600" width="1000" font-family="-apple-system,Segoe UI,Helvetica,Arial,sans-serif">
  <style>
    .ns{rx:9;ry:9;stroke-width:1.5}
    .pill{fill:#ffffff;stroke:#cbd5e1;stroke-width:1;rx:6;ry:6}
    .t{font-size:12.5px;fill:#0f172a}
    .nt{font-size:12.5px;font-weight:700}
    .sub{font-size:10.5px;fill:#475569}
    .lbl{font-size:11px;fill:#334155}
    .hbone{stroke:#7c3aed;stroke-width:2.4;fill:none;marker-end:url(#p);marker-start:url(#ps)}
  </style>
  <defs>
    <marker id="p" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#7c3aed"/></marker>
    <marker id="ps" markerWidth="9" markerHeight="9" refX="2" refY="3" orient="auto"><path d="M7,0 L0,3 L7,6 Z" fill="#7c3aed"/></marker>
  </defs>

  <text x="20" y="26" font-size="15" font-weight="700" fill="#0f172a">Solo Enterprise agentgateway across two ambient clusters, peered over HBONE</text>
  <text x="20" y="44" class="sub">north-south ingress · cross-cluster failover · MCP tool federation — watch it in the Gloo UI service graph</text>

  <!-- EAST cluster -->
  <rect x="20" y="58" width="450" height="500" rx="12" fill="#f8fafc" stroke="#1e293b" stroke-width="2"/>
  <text x="38" y="80" class="nt" fill="#1e293b">kind-east-ag</text>
  <text x="128" y="80" class="sub">pods 10.10.0.0/16 · network east-ag</text>

  <rect x="38" y="92" width="414" height="88" class="ns" fill="#ede9fe" stroke="#6d28d9"/>
  <text x="52" y="111" class="nt" fill="#6d28d9">agentgateway-system</text>
  <rect x="52" y="120" width="190" height="48" class="pill"/><text x="62" y="140" class="t">north-south ingress</text><text x="62" y="157" class="sub">class enterprise-agentgateway</text>
  <rect x="252" y="120" width="190" height="48" class="pill"/><text x="262" y="140" class="t">MCP endpoint /mcp</text><text x="262" y="157" class="sub">federates catalog + orders</text>

  <rect x="38" y="190" width="414" height="72" class="ns" fill="#e0f2fe" stroke="#0369a1"/>
  <text x="52" y="209" class="nt" fill="#0369a1">bookinfo</text>
  <rect x="52" y="217" width="146" height="38" class="pill"/><text x="62" y="240" class="t">productpage (global)</text>
  <rect x="208" y="217" width="118" height="38" class="pill"/><text x="218" y="240" class="t">reviews v1-v3</text>
  <rect x="336" y="217" width="116" height="38" class="pill"/><text x="346" y="240" class="t">details · ratings</text>

  <rect x="38" y="272" width="204" height="72" class="ns" fill="#dcfce7" stroke="#166534"/>
  <text x="52" y="291" class="nt" fill="#166534">ai-tools</text>
  <rect x="52" y="299" width="176" height="38" class="pill"/><text x="62" y="322" class="t">catalog-mcp (local)</text>
  <rect x="252" y="272" width="200" height="72" class="ns" fill="#fef3c7" stroke="#92400e"/>
  <text x="266" y="291" class="nt" fill="#92400e">ai-agents</text>
  <rect x="266" y="299" width="172" height="38" class="pill"/><text x="276" y="322" class="t">dev-ui (MCP client)</text>

  <rect x="38" y="474" width="414" height="70" class="ns" fill="#f1f5f9" stroke="#475569"/>
  <text x="52" y="494" class="nt" fill="#334155">istio-eastwest — HBONE east-west gateway</text>
  <text x="52" y="512" class="sub">ztunnel-backed · LoadBalancer (MetalLB) · :15008</text>
  <text x="52" y="530" class="sub">istiod-gloo control plane · shared root CA</text>

  <!-- WEST cluster -->
  <rect x="530" y="58" width="450" height="500" rx="12" fill="#f8fafc" stroke="#1e293b" stroke-width="2"/>
  <text x="548" y="80" class="nt" fill="#1e293b">kind-west-ag</text>
  <text x="638" y="80" class="sub">pods 10.20.0.0/16 · network west-ag</text>

  <rect x="548" y="92" width="414" height="88" class="ns" fill="#f8fafc" stroke="#94a3b8" stroke-dasharray="4 3"/>
  <text x="562" y="120" class="lbl">Peer cluster — no ingress of its own. It publishes its</text>
  <text x="562" y="138" class="lbl">global services (productpage, orders-mcp) to east</text>
  <text x="562" y="156" class="lbl">over the HBONE fabric; east routes to them by hostname.</text>

  <rect x="548" y="190" width="414" height="72" class="ns" fill="#e0f2fe" stroke="#0369a1"/>
  <text x="562" y="209" class="nt" fill="#0369a1">bookinfo</text>
  <rect x="562" y="217" width="146" height="38" class="pill"/><text x="572" y="240" class="t">productpage (global)</text>
  <rect x="718" y="217" width="118" height="38" class="pill"/><text x="728" y="240" class="t">reviews v1-v3</text>
  <rect x="846" y="217" width="106" height="38" class="pill"/><text x="856" y="240" class="t">details</text>

  <rect x="548" y="272" width="414" height="72" class="ns" fill="#dcfce7" stroke="#166534"/>
  <text x="562" y="291" class="nt" fill="#166534">ai-tools</text>
  <rect x="562" y="299" width="230" height="38" class="pill"/><text x="572" y="322" class="t">orders-mcp (global service)</text>

  <rect x="548" y="474" width="414" height="70" class="ns" fill="#f1f5f9" stroke="#475569"/>
  <text x="562" y="494" class="nt" fill="#334155">istio-eastwest — HBONE east-west gateway</text>
  <text x="562" y="512" class="sub">ztunnel-backed · LoadBalancer (MetalLB) · :15008</text>
  <text x="562" y="530" class="sub">istiod-gloo control plane · shared root CA</text>

  <!-- HBONE double arrow between the two east-west GWs -->
  <path class="hbone" d="M456,509 L526,509"/>
  <text x="491" y="490" class="lbl" text-anchor="middle" fill="#7c3aed" font-weight="700">HBONE</text>
  <text x="491" y="503" class="sub" text-anchor="middle" fill="#7c3aed">:15008</text>
</svg>

> One **agentgateway** on `east-ag` is the front door. Bookinfo runs on **both** clusters; `productpage` is a **global service**, so when east's copy goes away the mesh serves it from **west** over HBONE. Two MCP tool servers — `catalog-mcp` on east and `orders-mcp` on west — are federated behind a single agentgateway **`/mcp`** endpoint, and one `AgentgatewayPolicy` scopes which tools any caller sees. Watch the cross-cluster hops light up in the **Gloo UI service graph**.

## Open the consoles

Run this once and leave it in a terminal — it port-forwards the Gloo UI (the multicluster service graph) and prints the console URLs:

```bash
./demo-scripts/open-consoles.sh
```

| Console | URL | What it's for |
|---|---|---|
| **Gloo UI** | **http://localhost:8090** | service graph across **both** clusters — the cross-cluster hops show up here |
| Enterprise UI (agentgateway) | http://\<enterprise-ui LB\> | agentgateway dashboard + tracing (printed by `open-consoles.sh`) |

> The Gloo UI is the one to keep open. As you run each exercise below, refresh the **service graph** and watch traffic cross from `east-ag` to `west-ag`.

## Connect and deploy the workloads

`connect.sh` sets the two cluster contexts and confirms peering. `deploy-workloads.sh` lays down Bookinfo on both clusters plus the two MCP tool servers (idempotent — safe to re-run).

In [1]:
[ -d agentgw-multi-cluster-kind ] && cd agentgw-multi-cluster-kind || :
source demo-scripts/connect.sh

east = kind-east-ag   west = kind-west-ag


✓ clusters peered over HBONE


In [2]:
./demo-scripts/deploy-workloads.sh

══> Bookinfo on both clusters (ambient + network label)


namespace/bookinfo created


   • [kind-east-ag] bookinfo applied


namespace/bookinfo created


   • [kind-west-ag] bookinfo applied


...

namespace/ai-tools created


══> dev-ui caller on east


namespace/ai-agents created


══> Wait for everything Ready (first MCP pull can take ~90s)


   ✓ workloads ready on both clusters


## 1. Reach a service through the agentgateway ingress

Expose Bookinfo's `productpage` through an agentgateway `Gateway` (class `enterprise-agentgateway`) and an `HTTPRoute`. This is the north-south front door — one agentgateway data plane on `east-ag`, reached on a MetalLB LoadBalancer IP.

In [3]:
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata: { name: bookinfo-gateway, namespace: bookinfo }
spec:
  gatewayClassName: enterprise-agentgateway
  listeners:
  - { name: http, port: 8080, protocol: HTTP, allowedRoutes: { namespaces: { from: Same } } }
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: productpage, namespace: bookinfo }
spec:
  parentRefs: [{ name: bookinfo-gateway }]
  rules:
  - backendRefs: [{ name: productpage, port: 9080 }]
EOF

kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Programmed gateway/bookinfo-gateway --timeout=90s
kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Ready \
  pod -l gateway.networking.k8s.io/gateway-name=bookinfo-gateway --timeout=120s

gateway.gateway.networking.k8s.io/bookinfo-gateway created


httproute.gateway.networking.k8s.io/productpage created


gateway.gateway.networking.k8s.io/bookinfo-gateway condition met


pod/bookinfo-gateway-6bbc46767d-kdvvg condition met


In [4]:
GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
echo "agentgateway ingress: http://$GW:8080/productpage"
curl -s -o /dev/null -w "  HTTP %{http_code}\n" "http://$GW:8080/productpage"    # → HTTP 200

agentgateway ingress: http://192.168.97.105:8080/productpage


  HTTP 200


**In the Gloo UI** (http://localhost:8090) open the **Graph** and select the `east-ag` cluster + `bookinfo` namespace — the `bookinfo-gateway` now fronts `productpage`. Everything so far is single-cluster; the next exercise makes it span clusters.

## 2. Fail `productpage` over to the other cluster — over HBONE

Label `productpage` a **global** service on both clusters. Solo Istio then publishes a cluster-wide hostname `productpage.bookinfo.mesh.internal` backed by a synthetic VIP whose endpoints span **both** clusters. Scale east's copy to zero and the mesh keeps serving it from **west** over HBONE — no client or DNS change.

In [5]:
for CTX in $CLUSTER1 $CLUSTER2; do
  kubectl --context $CTX -n bookinfo label svc productpage solo.io/service-scope=global --overwrite
done
sleep 8
istioctl --context $CLUSTER1 multicluster check 2>/dev/null | grep -i "Shared Services"   # → globally shared service(s)

service/productpage labeled


service/productpage labeled


✅ Shared Services Check: 3 globally shared service(s)


In [6]:
# Take east's productpage away
kubectl --context $CLUSTER1 -n bookinfo scale deploy productpage-v1 --replicas=0
kubectl --context $CLUSTER1 -n bookinfo wait --for=delete pod -l app=productpage --timeout=40s

# The global VIP still has an endpoint (west); the local service has none
istioctl --context $CLUSTER1 ztunnel-config service 2>/dev/null | grep -E "NAMESPACE|productpage" | head -4

deployment.apps/productpage-v1 scaled


pod/productpage-v1-95fd4795b-cmvt9 condition met


NAMESPACE           SERVICE NAME                                                  SERVICE VIP            WAYPOINT       ENDPOINTS


bookinfo            autogen.bookinfo.productpage                                  240.240.0.12,2001:2::c None           1/1


bookinfo            productpage                                                   10.96.59.241           None           0/0


bookinfo            productpage-v1                                                10.96.105.240          None           0/0


In [7]:
# From an in-mesh pod, the GLOBAL hostname fails over to west (HTTP 200)
kubectl --context $CLUSTER1 -n ai-agents exec deploy/dev-ui -- \
  curl -sS -o /dev/null -w "  GLOBAL (mesh.internal): HTTP %{http_code} via %{remote_ip}\n" -m 10 \
  http://productpage.bookinfo.mesh.internal:9080/productpage

# The plain short name has NO remote endpoints — it does not fail over
kubectl --context $CLUSTER1 -n ai-agents exec deploy/dev-ui -- \
  curl -sS -o /dev/null -m 6 -w "  SHORT   (local only):  HTTP %{http_code}\n" \
  http://productpage.bookinfo:9080/productpage \
  || echo "  SHORT   (local only):  no route — the short name does not fail over (expected)" 

  GLOBAL (mesh.internal): HTTP 200 via 240.240.0.12


curl: (52) Empty reply from server


  SHORT   (local only):  HTTP 000


command terminated with exit code 52


  SHORT   (local only):  no route — the short name does not fail over (expected)


In [8]:
# Restore east's productpage
kubectl --context $CLUSTER1 -n bookinfo scale deploy productpage-v1 --replicas=1
kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Ready pod -l app=productpage --timeout=90s

deployment.apps/productpage-v1 scaled


pod/productpage-v1-95fd4795b-krz6x condition met


**In the Gloo UI** service graph, with east's `productpage` scaled to zero you'll see the call cross from `east-ag` into `west-ag` — the request is served by the peer cluster over the HBONE east-west gateway. Failover is opt-in **by hostname**: only `*.mesh.internal` gets it, which is why the short name returned nothing.

## 3. Federate MCP tool servers from both clusters behind one endpoint

`catalog-mcp` runs on **east**, `orders-mcp` runs on **west** (a global service). One agentgateway `AgentgatewayBackend` federates both into a single MCP `/mcp` endpoint. East reaches the west server over HBONE via its global VIP, so a single `tools/list` returns tools from **both** clusters, tool names namespaced by target.

In [9]:
# The west orders-mcp is a global service; grab the VIP east routes to it on
VIP=$(istioctl --context $CLUSTER1 ztunnel-config service 2>/dev/null | awk '/orders-mcp/{print $3}' | cut -d, -f1)
echo "west orders-mcp reachable from east at: $VIP:3001 (over HBONE)"

kubectl --context $CLUSTER1 -n ai-tools apply -f - <<EOF
apiVersion: agentgateway.dev/v1alpha1
kind: AgentgatewayBackend
metadata: { name: mcp-be }
spec:
  mcp:
    targets:
    - { name: catalog, static: { host: catalog-mcp.ai-tools.svc.cluster.local, port: 3001, protocol: SSE } }
    - { name: orders,  static: { host: "$VIP", port: 3001, protocol: SSE } }   # west, over HBONE
EOF

kubectl --context $CLUSTER1 -n ai-tools apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata: { name: mcp-gateway, namespace: ai-tools }
spec:
  gatewayClassName: enterprise-agentgateway
  listeners: [{ name: http, port: 8080, protocol: HTTP, allowedRoutes: { namespaces: { from: Same } } }]
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: mcp-route, namespace: ai-tools }
spec:
  parentRefs: [{ name: mcp-gateway }]
  rules:
  - backendRefs: [{ group: agentgateway.dev, kind: AgentgatewayBackend, name: mcp-be }]
EOF

kubectl --context $CLUSTER1 -n ai-tools wait --for=condition=Programmed gateway/mcp-gateway --timeout=90s
kubectl --context $CLUSTER1 -n ai-tools wait --for=condition=Ready \
  pod -l gateway.networking.k8s.io/gateway-name=mcp-gateway --timeout=120s

west orders-mcp reachable from east at: 240.240.0.11:3001 (over HBONE)


agentgatewaybackend.agentgateway.dev/mcp-be created


gateway.gateway.networking.k8s.io/mcp-gateway created


httproute.gateway.networking.k8s.io/mcp-route created


gateway.gateway.networking.k8s.io/mcp-gateway condition met


pod/mcp-gateway-6b785694fb-nqs84 condition met


In [10]:
# One MCP endpoint, tools from BOTH clusters (catalog_* = east, orders_* = west)
./demo-scripts/mcp.sh list
echo "---"
./demo-scripts/mcp.sh list | sed 's/_.*//' | sort | uniq -c

catalog_echo


catalog_get-annotated-message


catalog_get-env


catalog_get-resource-links


catalog_get-resource-reference


...

orders_toggle-subscriber-updates


orders_trigger-long-running-operation


---


  13 catalog


  13 orders


In [11]:
# Call a tool that runs on WEST (over HBONE), and one that runs locally on EAST
echo "orders_echo   (west):  $(./demo-scripts/mcp.sh call orders_echo '{"message":"hi from east"}')"
echo "catalog_get-sum (east): $(./demo-scripts/mcp.sh call catalog_get-sum '{"a":40,"b":2}')" 

orders_echo   (west):  Echo: hi from east


catalog_get-sum (east): The sum of 40 and 2 is 42.


One agentgateway endpoint, one MCP session, tools drawn from **two clusters**. The `orders_*` calls execute on `west-ag` and return over HBONE; the caller never knew there were two clusters.

## 4. Govern the federated tools with one policy

An `AgentgatewayPolicy` with `action: Allow` is deny-by-default: only tools matching the CEL rules survive `tools/list`, and blocked tools can't be called — across **both** clusters, from one policy.

In [12]:
kubectl --context $CLUSTER1 -n ai-tools apply -f - <<'EOF'
apiVersion: agentgateway.dev/v1alpha1
kind: AgentgatewayPolicy
metadata: { name: mcp-tool-allowlist, namespace: ai-tools }
spec:
  targetRefs: [{ group: agentgateway.dev, kind: AgentgatewayBackend, name: mcp-be }]
  backend:
    mcp:
      authorization:
        action: Allow            # deny-by-default: only matching tools pass
        policy:
          matchExpressions:
          - 'mcp.tool.name == "echo"'
          - 'mcp.tool.name == "get-sum"'
EOF
sleep 3

agentgatewaypolicy.agentgateway.dev/mcp-tool-allowlist created


In [13]:
# The federated list collapses to the allowed tools on BOTH targets
./demo-scripts/mcp.sh list

# A blocked tool is filtered out entirely — it can't even be called
echo "blocked orders_get-env → $(./demo-scripts/mcp.sh call orders_get-env '{}' 2>&1 | head -1)" 

catalog_echo


catalog_get-sum


orders_echo


orders_get-sum


blocked orders_get-env → Unknown tool: orders_get-env


That's the multicluster story end to end: one agentgateway ingress, cross-cluster failover over HBONE, MCP tools federated from two clusters behind one endpoint, and a single policy governing them all — with the traffic visible in the Gloo UI service graph.

## Reset

Remove the exercise workloads (the platform standup stays up):

```bash
./demo-scripts/reset.sh
```